In [ ]:
import joblib
import numpy as np
import pandas as pd
import sqlite3
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def load_data():
  # Connect to Regression.db and load a random sample of 10% for quick testing
  conn = sqlite3.connect("regression.db")
  query = "SELECT * FROM electric_motor_temperature ORDER BY RANDOM() LIMIT 133000"
  df = pd.read_sql(query, conn)
  conn.close()
  return df


def preprocess_and_feature_engineering(df, scaler=None, fit_scaler=False):
  """Applies consistent data processing and feature creation steps."""
  if "timestamp" in df.columns:
    df = df.drop(columns=["timestamp"])

  if "u_q" in df.columns and "u_d" in df.columns:
    df["u_magnitude"] = np.sqrt(df["u_q"] ** 2 + df["u_d"] ** 2)

  target_col = "pm"  # Permanent magnet temperature
  X = df.drop(columns=[target_col])
  y = df[target_col]

  if fit_scaler:
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
    joblib.dump(scaler, "scaler.pkl")
  else:
    X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns)

  return X_scaled, y


def main():
  print("Loading data sample from database...")
  df = load_data()

  print(
      f"Loaded sample shape: {df.shape}. Preprocessing and engineering"
      " features..."
  )
  X, y = preprocess_and_feature_engineering(df, fit_scaler=True)

  # Train-test split
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, random_state=42
  )

  print("Training model...")
  model = RandomForestRegressor(
      n_estimators=50,  # Optimized for faster training
      max_depth=15,  # Limited depth to speed up execution
      random_state=42,
      n_jobs=-1,  # Use all available CPU cores
  )
  model.fit(X_train, y_train)

  # Evaluation
  preds = model.predict(X_test)
  print(f"R2 Score: {r2_score(y_test, preds):.4f}")
  print(f"MAE: {mean_absolute_error(y_test, preds):.4f}")
  print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.4f}")

  # Save model
  joblib.dump(model, "pmsm_model.pkl")
  print("Model saved successfully as pmsm_model.pkl")


if __name__ == "__main__":
  main()

Loading data from database...
Preprocessing and engineering features...
Training model...
